# 02 LLM 初识：Transformer、Attention 与生成机制

[TOC]

读者：准备学习 AI Agent 开发，需要建立 LLM 基础模型的工程师。

前置知识：Python 函数、列表、字典和基础概率。

学习结果：能够解释文本如何变成 token、Transformer 如何处理上下文、模型如何训练并逐 token 生成答案。

学习路径：输入表示 -> Transformer -> Attention -> 生成 -> 训练 -> 工程约束。

本章不调用外部 API。所有实验使用 Python 标准库。

## 结论

LLM 的基础工作流：

```text
文本
-> tokenizer 编码为 token ID
-> token embedding + 位置信息
-> 多层 Transformer 生成 hidden state
-> LM Head 输出整个词表的 logits
-> softmax 得到概率分布
-> sampling 选出下一个 token ID
-> tokenizer 解码为文本
```

需要掌握以下概念：

1. token：模型处理文本的基本单位。
2. token ID：token 在词表中的整数编号。
3. embedding：token ID 对应的向量表示。
4. attention：计算每个位置应该读取哪些上下文位置。
5. Transformer：堆叠 attention 和前馈网络的模型结构。
6. hidden state：Transformer 为每个位置计算出的上下文表示。
7. next-token prediction：根据最后位置的 hidden state 预测下一个 token。
8. KV Cache：缓存历史 token 的 Key 和 Value，减少重复计算。
9. loss：预测分布和目标 token 的差距。
10. weights：训练完成后保存的模型参数。

## 必备概念

| 概念 | 工程含义 |
| --- | --- |
| token | 文本进入模型前的切分单位 |
| token ID | token 在词表中的整数编号 |
| tokenizer | 把文本转成 token 的组件 |
| embedding | 把 token 转成向量 |
| position | token 在序列中的位置 |
| attention | 给上下文位置分配权重 |
| Transformer | 多层 attention + 前馈网络 |
| hidden state | 每个位置经过 Transformer 后的向量 |
| logits | 模型输出的未归一化分数 |
| softmax | 把 logits 转成概率分布 |
| sampling | 从概率分布中选择下一个 token |
| context window | 输入和输出加起来的 token 上限 |
| hallucination | 输出与事实、上下文或工具结果不一致 |
| training pair | 用当前位置预测下一个 token 的训练样本 |
| loss | 模型预测和目标 token 的差距 |
| batch | 一次并行计算的一组训练样本 |
| step | 一次参数更新 |
| epoch | 全量训练数据被使用一遍 |
| checkpoint | 某个训练时刻保存的参数和训练状态 |
| KV Cache | 推理时缓存历史位置的 Key 和 Value |
| weights | 模型训练后的参数 |

## 实验 1：Tokenization

目标：理解文本进入模型前会被切成 token。

说明：这里实现的是简化 tokenizer，用于理解机制。真实模型通常使用 BPE、Unigram、SentencePiece 等算法。

In [1]:
import re  # re 是 Python 标准库中的正则表达式模块。


def simple_tokenize(text: str) -> list[str]:
    # -> list[str] 表示函数返回字符串列表。
    # 规则依次匹配：英文单词、数字、中文单字、其他非空白字符。
    pattern = r"[A-Za-z]+(?:'[A-Za-z]+)?|\d+(?:\.\d+)?|[\u4e00-\u9fff]|[^\s]"
    return re.findall(pattern, text)  # findall 返回所有匹配结果。


text = "AI Agent 可以读取 README.md，并计算 16 的平方根。"
tokens = simple_tokenize(text)  # 调用函数，把原始文本切成 token。

print(tokens)
print("token_count =", len(tokens))


['AI', 'Agent', '可', '以', '读', '取', 'README', '.', 'md', '，', '并', '计', '算', '16', '的', '平', '方', '根', '。']
token_count = 19


In [2]:
# assert 用于验证结果。条件为 False 时，Python 会立即报错。
assert "AI" in tokens
assert "Agent" in tokens
assert "16" in tokens
assert len(tokens) > 10

print("tokenization checks passed")


tokenization checks passed


当前边界：

1. 中文被按单字切分。
2. 英文按单词切分。
3. 真实 tokenizer 会根据训练得到的词表切分。
4. 同一句话在不同模型中 token 数可能不同。

### 1.1 Token ID、Embedding 与位置

tokenizer 先把 token 映射为 token ID。模型再读取 token embedding，并加入位置信息。

```text
token
-> token ID
-> token embedding
+ position embedding
-> 输入 Transformer 的向量
```

位置信息让模型区分顺序。`用户 调用 工具` 和 `工具 调用 用户` 包含相同 token，位置不同，含义也不同。真实模型可使用学习得到的位置向量或 RoPE。

In [3]:
sample_tokens = ["AI", "Agent", "调用", "工具"]

# enumerate 每次返回两个值：位置编号和该位置的 token。
sample_vocab = {}
for token_id, token in enumerate(sample_tokens):
    sample_vocab[token] = token_id
# 根据词表，把 token 逐个转换成整数 ID。
token_ids = []
for token in sample_tokens:
    token_ids.append(sample_vocab[token])

# 二维向量用于展示查表和相加。真实 embedding 通常有数千维。
token_embeddings = {
    0: [0.8, 0.1],
    1: [0.7, 0.3],
    2: [0.2, 0.9],
    3: [0.1, 0.8],
}
position_embeddings = [
    [0.00, 0.00],
    [0.05, 0.01],
    [0.10, 0.02],
    [0.15, 0.03],
]

# 每个位置的最终输入 = token 向量 + 位置向量。
input_vectors = []
for position, token_id in enumerate(token_ids):
    token_vector = token_embeddings[token_id]
    position_vector = position_embeddings[position]

    # zip 把两个向量相同位置上的数字配成一对。
    combined = []
    for token_value, position_value in zip(token_vector, position_vector):
        combined.append(round(token_value + position_value, 3))

    input_vectors.append(combined)
    print(position, sample_tokens[position], token_id, combined)

assert token_ids == [0, 1, 2, 3]
assert input_vectors[1] != token_embeddings[1]
print("embedding checks passed")


0 AI 0 [0.8, 0.1]
1 Agent 1 [0.75, 0.31]
2 调用 2 [0.3, 0.92]
3 工具 3 [0.25, 0.83]
embedding checks passed


## 实验 2：Next Token Prediction

目标：理解模型生成文本时会反复预测下一个 token。

做法：用小语料统计 bigram 概率，模拟“看到上一个 token 后，下一个 token 的概率分布”。

In [4]:
from collections import Counter, defaultdict


corpus = [
    ["AI", "Agent", "调用", "工具"],
    ["AI", "Agent", "查询", "知识库"],
    ["AI", "Agent", "维护", "状态"],
    ["AI", "Agent", "调用", "函数"],
    ["用户", "提出", "任务"],
    ["模型", "预测", "下一个", "token"],
]

# 外层 key 是当前 token，内层 Counter 统计下一个 token 出现了几次。
# defaultdict(Counter) 会在 key 首次出现时自动创建空 Counter。
bigram_counts: dict[str, Counter[str]] = defaultdict(Counter)

# zip(sentence, sentence[1:]) 把相邻 token 配成训练对。
# 例如 [AI, Agent, 调用] 会得到 (AI, Agent)、(Agent, 调用)。
for sentence in corpus:
    for current_token, next_token in zip(sentence, sentence[1:]):
        bigram_counts[current_token][next_token] += 1


def next_token_distribution(current_token: str) -> dict[str, float]:
    counts = bigram_counts[current_token]
    total = sum(counts.values())
    probabilities = {}
    for token, count in counts.items():
        probabilities[token] = count / total
    return probabilities


distribution = next_token_distribution("Agent")
print(distribution)


{'调用': 0.5, '查询': 0.25, '维护': 0.25}


In [5]:
# Agent 后面共出现 4 次：调用 2 次，查询和维护各 1 次。
assert distribution["调用"] == 0.5
assert distribution["查询"] == 0.25
assert distribution["维护"] == 0.25
assert abs(sum(distribution.values()) - 1.0) < 1e-9

print("next-token checks passed")


next-token checks passed


当前边界：

1. bigram 只看上一个 token。
2. LLM 会使用更长上下文。
3. LLM 输出的是巨大词表上的概率分布。
4. 这个实验只用于理解“预测下一个 token”的形式。

## 实验 3：Temperature Sampling

目标：理解 temperature 如何影响随机性。

规则：

1. temperature 越低，分布越尖锐。
2. temperature 越高，分布越平。
3. temperature 接近 0 时，结果接近固定选择最高分 token。

In [6]:
import math
import random


logits = {
    "直接回答": 3.0,
    "查询知识库": 2.2,
    "调用工具": 1.8,
    "转人工": 0.4,
}


def softmax_with_temperature(scores: dict[str, float], temperature: float) -> dict[str, float]:
    # temperature 必须大于 0，提前检查可以避免除零错误。
    if temperature <= 0:
        raise ValueError("temperature must be positive")

    # 第一步：用 temperature 缩放每个 token 的分数。
    scaled = {}
    for token, score in scores.items():
        scaled[token] = score / temperature

    # 减去最大值可以避免 exp 结果过大。
    max_score = max(scaled.values())
    exp_scores = {}
    for token, score in scaled.items():
        exp_scores[token] = math.exp(score - max_score)

    # 第二步：除以总和，把所有分数转换成概率。
    total = sum(exp_scores.values())
    probabilities = {}
    for token, value in exp_scores.items():
        probabilities[token] = value / total
    return probabilities


def probability_value(item: tuple[str, float]) -> float:
    # 字典的 items() 元素形如 (token, probability)，索引 1 取概率。
    return item[1]


for temperature in [0.2, 0.7, 1.0, 2.0]:
    probs = softmax_with_temperature(logits, temperature)
    print("temperature =", temperature)
    # key 指定排序依据，reverse=True 表示概率从大到小。
    for token, probability in sorted(probs.items(), key=probability_value, reverse=True):
        print(f"  {token}: {probability:.3f}")
    print()


temperature = 0.2
  直接回答: 0.980
  查询知识库: 0.018
  调用工具: 0.002
  转人工: 0.000

temperature = 0.7
  直接回答: 0.656
  查询知识库: 0.209
  调用工具: 0.118
  转人工: 0.016

temperature = 1.0
  直接回答: 0.548
  查询知识库: 0.246
  调用工具: 0.165
  转人工: 0.041

temperature = 2.0
  直接回答: 0.401
  查询知识库: 0.269
  调用工具: 0.220
  转人工: 0.109



In [7]:
def sample_token(probabilities: dict[str, float], seed: int) -> str:
    # 固定 seed 后，每次运行会得到相同的随机序列，便于复现实验。
    rng = random.Random(seed)
    threshold = rng.random()
    cumulative = 0.0

    # 把概率依次累加，随机数落在哪个区间就选择哪个 token。
    for token, probability in probabilities.items():
        cumulative += probability
        if threshold <= cumulative:
            return token

    # 浮点误差可能让累计值略小于 1，此时返回最后一个 token。
    last_token = ""
    for token in probabilities:
        last_token = token
    return last_token


low_temp_probs = softmax_with_temperature(logits, 0.2)
high_temp_probs = softmax_with_temperature(logits, 2.0)

low_temperature_samples = []
high_temperature_samples = []
for seed in range(10):
    low_temperature_samples.append(sample_token(low_temp_probs, seed))
    high_temperature_samples.append(sample_token(high_temp_probs, seed))

print("low temperature samples:")
print(low_temperature_samples)

print("high temperature samples:")
print(high_temperature_samples)


low temperature samples:
['直接回答', '直接回答', '直接回答', '直接回答', '直接回答', '直接回答', '直接回答', '直接回答', '直接回答', '直接回答']
high temperature samples:
['调用工具', '直接回答', '转人工', '直接回答', '直接回答', '查询知识库', '调用工具', '直接回答', '直接回答', '查询知识库']


In [8]:
# 低温度让最高分 token 的概率更集中，高温度让低分 token 获得更多概率。
assert low_temp_probs["直接回答"] > high_temp_probs["直接回答"]
assert high_temp_probs["转人工"] > low_temp_probs["转人工"]
assert abs(sum(low_temp_probs.values()) - 1.0) < 1e-9
assert abs(sum(high_temp_probs.values()) - 1.0) < 1e-9

print("temperature checks passed")


temperature checks passed


当前边界：

1. 这里只演示 temperature。
2. 真实生成还会使用 top-k、top-p、frequency penalty、presence penalty 等策略。
3. 随机性影响表达方式，也可能影响事实稳定性。

## Transformer 的数据流

Decoder-only LLM 会堆叠多层 Transformer Block。每层都接收一组位置向量，并输出同样数量的新向量。

| 部件 | 作用 |
| --- | --- |
| Token Embedding | 把 token ID 映射为向量 |
| Position / RoPE | 注入顺序信息 |
| Self-Attention | 从已有上下文位置读取信息 |
| Feed Forward Network | 对每个位置做非线性变换 |
| Residual Connection | 把输入与子层输出相加 |
| LayerNorm | 稳定数值分布 |
| LM Head | 把最后位置的 hidden state 映射到词表 logits |

预测下一个 token 的主链路：

```text
最后一个已有 token 的 hidden state
-> LM Head
-> 整个词表的 logits
-> softmax / sampling
-> 下一个 token
```

下一个 token 尚未进入模型，因此它没有 Query。模型使用最后一个已有位置的 hidden state 预测它。

## 实验 4：Attention 权重

目标：理解 Q/K/V、权重计算、逐 token 生成的关系。

Self-Attention 的输入是一组位置向量。每个位置经过矩阵变换生成 Q、K、V：

| 向量 | 工程含义 | 类比 |
| --- | --- | --- |
| Query | 当前位置要读取什么信息 | 查询条件 |
| Key | 每个上下文位置提供的匹配特征 | 索引标签 |
| Value | 每个上下文位置携带的内容 | 可读取内容 |

权重计算流程：

```text
X -> Q = XWq, K = XWk, V = XWv
-> score = QK^T / sqrt(dk)
-> causal mask 屏蔽未来位置
-> softmax(score)
-> 得到 attention weights
-> 对所有 value 加权求和
-> 得到当前位置的上下文向量
```

生成下一个 token 时，模型读取最后一个已有位置的 hidden state。该位置在每层中通过 Query 读取自己和前文。新 token 追加后成为新的最后位置。

### 4.1 Q/K/V 对应到用户请求

用户请求：

```text
请 根据 会议 纪要 提取 待办
```

任务：使用最后一个已有 token `待办` 所在位置的 Query，计算它对所有可见位置的 Attention 权重。该位置完成多层计算后，LM Head 再预测下一个 token。

示例把 Query 设为偏向“会议待办提取”的二维向量，因此 `会议`、`纪要`、`待办` 会获得较高权重。

In [9]:
def dot(left: list[float], right: list[float]) -> float:
    # 点积：对应位置相乘，再把结果相加。
    result = 0.0
    for left_value, right_value in zip(left, right):
        result += left_value * right_value
    return result


def softmax(values: list[float]) -> list[float]:
    max_value = max(values)
    exp_values = []
    for value in values:
        exp_values.append(math.exp(value - max_value))
    total = sum(exp_values)

    probabilities = []
    for value in exp_values:
        probabilities.append(value / total)
    return probabilities


prompt_tokens = ["请", "根据", "会议", "纪要", "提取", "待办"]

# 二维向量只用于演示。
# 第 1 维表示“会议/待办相关”，第 2 维表示“通用表达相关”。
key_vectors = {
    "请": [0.1, 0.8],
    "根据": [0.2, 0.7],
    "会议": [0.9, 0.2],
    "纪要": [0.8, 0.3],
    "提取": [0.6, 0.6],
    "待办": [1.0, 0.1],
}

value_vectors = {
    "请": [0.0, 1.0],
    "根据": [0.1, 0.9],
    "会议": [0.9, 0.2],
    "纪要": [0.8, 0.3],
    "提取": [0.6, 0.6],
    "待办": [1.0, 0.1],
}

# 模拟最后位置“待办”经过 Wq 后得到的 Query。
query_from_last_position = [1.0, 0.2]
scale = math.sqrt(len(query_from_last_position))
print("scale:",scale)
# 每个列表元素代表一个位置，重复 token 也会保留为两行。
attention_rows = []
for position, token in enumerate(prompt_tokens):
    score = dot(query_from_last_position, key_vectors[token]) / scale
    attention_rows.append({"position": position, "token": token, "score": score})

scores = []
for row in attention_rows:
    scores.append(row["score"])
weights_list = softmax(scores)

# zip 把第 i 行和第 i 个权重配在一起。
for row, weight in zip(attention_rows, weights_list):
    row["weight"] = weight

print("position token score weight")
for row in attention_rows:
    print(
        f"{row['position']:>8} {row['token']:<5}"
        f" {row['score']:.3f} {row['weight']:.3f}"
    )


scale: 1.4142135623730951
position token score weight
       0 请     0.184 0.120
       1 根据    0.240 0.127
       2 会议    0.665 0.195
       3 纪要    0.608 0.184
       4 提取    0.509 0.167
       5 待办    0.721 0.206


In [10]:
# 上下文向量初始为 0，随后累加 weight * value。
context_vector = [0.0, 0.0]

for row in attention_rows:
    value = value_vectors[row["token"]]
    weight = row["weight"]
    context_vector[0] += weight * value[0]
    context_vector[1] += weight * value[1]

rounded_context = []
for value in context_vector:
    rounded_context.append(round(value, 3))
print("context_vector:", rounded_context)

# 当前示例没有重复 token，因此可以临时按 token 名称读取权重做断言。
weight_by_token = {}
for row in attention_rows:
    weight_by_token[row["token"]] = row["weight"]
assert weight_by_token["待办"] > weight_by_token["请"]
assert weight_by_token["会议"] > weight_by_token["根据"]
assert context_vector[0] > context_vector[1]

print("qkv attention checks passed")


context_vector: [0.642, 0.45]
qkv attention checks passed


### 4.2 Causal Mask

Decoder-only LLM 训练时会同时计算多个位置。位置 `i` 只能读取 `0..i`，不能读取未来 token。Causal Mask 保证训练过程和逐 token 推理使用相同的信息边界。

In [11]:
def causal_mask(size: int) -> list[list[int]]:
    mask = []

    # 每一行代表一个 Query 位置，每一列代表一个 Key 位置。
    for query_position in range(size):
        row = []
        for key_position in range(size):
            if key_position <= query_position:
                row.append(1)  # 允许读取当前位置和前文。
            else:
                row.append(0)  # 屏蔽未来位置。
        mask.append(row)

    return mask


mask = causal_mask(4)
for row in mask:
    print(row)

assert mask[0] == [1, 0, 0, 0]
assert mask[2] == [1, 1, 1, 0]
assert mask[3] == [1, 1, 1, 1]
print("causal mask checks passed")


[1, 0, 0, 0]
[1, 1, 0, 0]
[1, 1, 1, 0]
[1, 1, 1, 1]
causal mask checks passed


### 4.3 从上下文向量到下一个 token

Attention 得到的是上下文向量。模型后面还会经过若干层计算，最后通过 LM Head 得到词表 logits。

这里用一个简化函数模拟 LM Head：

```text
context_vector
-> candidate token logits
-> softmax
-> 选择下一个 token
```

In [12]:
def lm_head(context_vector: list[float]) -> dict[str, float]:
    # 序列解包：把二维向量的两个数字分别赋给两个变量。
    task_related, expression_related = context_vector

    # 每个候选 token 都使用一组不同系数计算 logit。
    return {
        "需要": 1.0 * task_related + 0.5 * expression_related,
        "天气": 0.1 * task_related + 1.2 * expression_related,
        "负责人": 0.7 * task_related + 0.1 * expression_related,
        "截止日期": 0.9 * task_related + 0.2 * expression_related,
    }


next_logits = lm_head(context_vector)
next_probs = softmax_with_temperature(next_logits, temperature=0.7)

print("next token probabilities:")
for token, probability in sorted(next_probs.items(), key=probability_value, reverse=True):
    print(f"  {token}: {probability:.3f}")

# max 默认比较字典的 key；key=next_probs.get 改为比较每个 key 对应的概率。
next_token = max(next_probs, key=next_probs.get)
print("selected next token:", next_token)

assert next_token in {"需要", "负责人", "截止日期"}
assert next_probs["天气"] < next_probs[next_token]

print("next-token from attention checks passed")


next token probabilities:
  需要: 0.330
  截止日期: 0.249
  天气: 0.227
  负责人: 0.194
selected next token: 需要
next-token from attention checks passed


### 4.4 生成一个 token 后继续生成

生成流程按 step 重复：

```text
step 1: prompt -> attention -> logits -> token_1
step 2: prompt + token_1 -> attention -> logits -> token_2
step 3: prompt + token_1 + token_2 -> attention -> logits -> token_3
```

每次新增 token 后，上下文都会变化。下一步 attention 会同时看到原始用户请求和已经生成的 token。

In [13]:
def attention_context(tokens: list[str], query: list[float]) -> tuple[list[dict], list[float]]:
    # 返回两个结果：每个位置的计算详情，以及最终上下文向量。
    scale = math.sqrt(len(query))
    rows = []
    for position, token in enumerate(tokens):
        # dict.get(key, default) 在 token 不存在时返回默认向量。
        key = key_vectors.get(token, [0.4, 0.4])
        rows.append({
            "position": position,
            "token": token,
            "score": dot(query, key) / scale,
        })

    scores = []
    for row in rows:
        scores.append(row["score"])
    weights_list = softmax(scores)
    for row, weight in zip(rows, weights_list):
        row["weight"] = weight

    context = [0.0, 0.0]
    for row in rows:
        value = value_vectors.get(row["token"], [0.4, 0.4])
        weight = row["weight"]
        context[0] += weight * value[0]
        context[1] += weight * value[1]

    return rows, context


def choose_next_token(tokens: list[str]) -> tuple[str, dict[str, float]]:
    query = [1.0, 0.2]
    # _ 表示第一个返回值在这里不使用，只保留 context。
    _, context = attention_context(tokens, query)
    logits = lm_head(context)

    # 已经输出“需要”后，更倾向输出“负责人”，模拟自回归生成中的上下文变化。
    # tokens[-1] 读取列表最后一个 token；tokens 为空时先跳过该判断。
    if tokens and tokens[-1] == "需要":
        logits["负责人"] += 0.8

    probs = softmax_with_temperature(logits, temperature=0.5)
    # 选择概率最高的 token，并把概率表一起返回。
    return max(probs, key=probs.get), probs


generated_tokens: list[str] = []
# copy 创建新列表，避免 append 修改原始 prompt_tokens。
context_tokens = prompt_tokens.copy()

for step in range(2):
    token, probs = choose_next_token(context_tokens)
    generated_tokens.append(token)
    context_tokens.append(token)

    print(f"step {step + 1}")
    print("  context:", context_tokens)
    print("  selected:", token)
    rounded_probs = {}
    for candidate, probability in probs.items():
        rounded_probs[candidate] = round(probability, 3)
    print("  probs:", rounded_probs)

print("generated:", generated_tokens)

assert len(generated_tokens) == 2
assert generated_tokens[0] == "需要"
assert generated_tokens[1] == "负责人"

print("autoregressive generation checks passed")


step 1
  context: ['请', '根据', '会议', '纪要', '提取', '待办', '需要']
  selected: 需要
  probs: {'需要': 0.365, '天气': 0.216, '负责人': 0.173, '截止日期': 0.245}
step 2
  context: ['请', '根据', '会议', '纪要', '提取', '待办', '需要', '负责人']
  selected: 负责人
  probs: {'需要': 0.213, '天气': 0.132, '负责人': 0.512, '截止日期': 0.144}
generated: ['需要', '负责人']
autoregressive generation checks passed


### 4.5 Prefill、Decode 与 KV Cache

推理分为两个阶段：

| 阶段 | 输入 | 计算 |
| --- | --- | --- |
| prefill | 完整 prompt | 计算所有 prompt 位置，并保存每层的 K/V |
| decode | 每次新增的一个 token | 计算新位置的 Q/K/V，读取历史 KV Cache |

KV Cache 省去历史位置的 K/V 投影。新 Query 仍需和所有历史 Key 计算 Attention，因此上下文越长，单个新 token 的解码成本越高。

In [14]:
prompt_length = 6
new_token_count = 4

# 只统计需要执行 K/V 投影的位置数量。
# 没有缓存时，每轮都要重新处理 prompt 和已经生成的 token。
without_kv_cache = 0
for step in range(new_token_count):
    context_length = prompt_length + step
    without_kv_cache += context_length

# 使用缓存后，prompt 处理一次，每个新 token 也只投影一次。
with_kv_cache = prompt_length + new_token_count

print("K/V projections without cache:", without_kv_cache)
print("K/V projections with cache:", with_kv_cache)

assert with_kv_cache < without_kv_cache
print("kv cache checks passed")


K/V projections without cache: 30
K/V projections with cache: 10
kv cache checks passed


### 4.6 练习：重复 token 对应不同位置

下面的上下文包含两个 `规则`。运行后检查两个位置是否都保留了独立权重。

In [15]:
exercise_tokens = ["请", "根据", "纪要", "纪要", "提取", "待办"]
# 第二个返回值是上下文向量，本练习只检查位置，所以用 _ 忽略它。
exercise_rows, _ = attention_context(exercise_tokens, query_from_last_position)

minute_rows = []
for row in exercise_rows:
    if row["token"] == "纪要":
        minute_rows.append(row)

for row in minute_rows:
    print({
        "position": row["position"],
        "token": row["token"],
        "weight": round(row["weight"], 3),
    })

assert len(exercise_rows) == len(exercise_tokens)
minute_positions = []
for row in minute_rows:
    minute_positions.append(row["position"])
assert minute_positions == [2, 3]
print("position-aware attention checks passed")


{'position': 2, 'token': '纪要', 'weight': 0.186}
{'position': 3, 'token': '纪要', 'weight': 0.186}
position-aware attention checks passed


当前边界：

1. 向量是手工设计的二维向量。
2. 真实模型通过训练得到的 Wq、Wk、Wv 计算 Q/K/V。
3. 真实模型包含多层、多头 Attention；不同头会学习不同关系。
4. Attention 权重属于某一层、某一头、某一位置，不能直接视为模型的完整解释。
5. 真实 LM Head 会对整个词表计算 logits。
6. 这里保留主流程：位置向量 -> Q/K/V -> 权重 -> hidden state -> logits -> 新 token。

## 实验 5：从训练数据到模型参数

目标：理解训练如何把文本样本变成模型参数。

代码使用只看前一个 token 的 toy bigram。它负责展示训练机制，不代表 Transformer 的参数结构。

训练阶段的核心流程：

```text
训练文本
训练文本 -> tokenizer -> token ID
-> input / label 序列
-> forward 得到每个位置的 logits
-> cross-entropy loss
-> backward 计算梯度
-> optimizer 更新参数
-> validation / checkpoint
```

训练和推理的区别：

| 单位 | 含义 |
| --- | --- |
| batch | 一次 forward/backward 使用的一组序列 |
| step | optimizer 执行一次参数更新 |
| epoch | 训练集完整使用一遍 |
| checkpoint | 保存某一步的参数、优化器状态和进度 |

常见训练阶段：

| 阶段 | 数据 | 目标 |
| --- | --- | --- |
| 预训练 | 大规模连续文本 | 学习语言和知识，预测下一个 token |
| SFT | 指令与回答 | 学习遵循指令和回答格式 |
| 偏好对齐 | 偏好数据或奖励信号 | 调整回答质量和行为边界 |

推理通常加载 tokenizer、模型配置和模型权重。恢复训练还需要 optimizer state、scheduler state、随机数状态和训练进度。

### 5.1 构造训练样本

Causal Language Modeling 将连续序列错开一位：

例子：

```text
AI Agent 调用 工具
```

```text
input  = [AI,    Agent, 调用]
label  = [Agent, 调用,  工具]
loss   = [L1,    L2,    L3]
```

Transformer 可以并行计算序列中多个位置的 loss。本实验将它拆成三个 bigram 训练对：

```text
input=AI      label=Agent
input=Agent   label=调用
input=调用    label=工具
```

bigram 每次只读取一个 token，便于直接观察参数如何变化。

In [16]:
training_sentences = [
    ["AI", "Agent", "调用", "工具"],
    ["AI", "Agent", "查询", "知识库"],
    ["AI", "Agent", "维护", "状态"],
    ["用户", "提交", "会议", "纪要"],
    ["会议", "纪要", "需要", "整理"],
    ["整理", "完成", "发送", "通知"],
]

# 每个训练对保存为 (当前 token, 目标 token)。
training_pairs: list[tuple[str, str]] = []

# sentence[1:] 删除第一个 token；zip 会把原序列和右移序列配对。
for sentence in training_sentences:
    for current_token, target_token in zip(sentence, sentence[1:]):
        training_pairs.append((current_token, target_token))

# 先用 set 去重，再排序，得到稳定的词表顺序。
vocab_set = set()
for current_token, target_token in training_pairs:
    vocab_set.add(current_token)
    vocab_set.add(target_token)
vocab = sorted(vocab_set)

print("vocab_size =", len(vocab))
print("first 8 training pairs:")
# [:8] 只取列表前 8 项，控制输出长度。
for pair in training_pairs[:8]:
    print(" ", pair)

assert ("AI", "Agent") in training_pairs
assert ("Agent", "调用") in training_pairs
assert "工具" in vocab


vocab_size = 17
first 8 training pairs:
  ('AI', 'Agent')
  ('Agent', '调用')
  ('调用', '工具')
  ('AI', 'Agent')
  ('Agent', '查询')
  ('查询', '知识库')
  ('AI', 'Agent')
  ('Agent', '维护')


### 5.2 Toy 模型的参数表

Toy 模型用二维字典保存参数：

```text
logits[current_token][candidate_next_token] = score
```

score 越高，candidate 越容易成为下一个 token。真实 LLM 的参数分布在 embedding、Attention、FFN、LayerNorm 和 LM Head 的矩阵中。

In [17]:
def make_empty_logits(vocab: list[str]) -> dict[str, dict[str, float]]:
    # 从外向内读：遍历每个 current，为它创建一张 candidate -> 0.0 的内层字典。
    # 外层 current 表示当前 token；内层 candidate 表示候选下一个 token。
    return {current: {candidate: 0.0 for candidate in vocab} for current in vocab}


trained_logits = make_empty_logits(vocab)

print("initial logits for Agent:")
agent_scores = {}
for token in ["调用", "查询", "维护"]:
    agent_scores[token] = trained_logits["Agent"][token]
print(agent_scores)

all_scores_are_zero = True
for score in trained_logits["Agent"].values():
    if score != 0.0:
        all_scores_are_zero = False
assert all_scores_are_zero


initial logits for Agent:
{'调用': 0.0, '查询': 0.0, '维护': 0.0}


### 5.3 一次训练 step

下面把 batch size 设为 1。每调用一次 `train_step`，就读取一个训练对并更新一次参数：

```text
读取一个训练样本
-> 当前参数给所有候选 token 打分
-> softmax 得到概率
-> 用目标 token 计算 loss
-> 提高目标 token 的 score
-> 降低其他 token 的 score
```

这里用简化梯度：

```text
gradient = probability - target_one_hot
logit = logit - learning_rate * gradient
```

In [18]:
def probabilities_for(logits_table: dict[str, dict[str, float]], current_token: str) -> dict[str, float]:
    # 先取出当前 token 对应的一行 logits，再转换成概率。
    return softmax_with_temperature(logits_table[current_token], temperature=1.0)


def cross_entropy_loss(probabilities: dict[str, float], target_token: str) -> float:
    # 目标 token 概率越高，-log(probability) 越小。
    # 1e-12 防止 probability 为 0 时出现 log(0)。
    return -math.log(probabilities[target_token] + 1e-12)


def train_step(
    logits_table: dict[str, dict[str, float]],
    current_token: str,
    target_token: str,
    learning_rate: float,
) -> float:
    # 1. 使用更新前的参数预测概率并计算 loss。
    probs = probabilities_for(logits_table, current_token)
    loss = cross_entropy_loss(probs, target_token)

    # 2. 依次更新这一行中的每个候选分数。
    for candidate_token, probability in probs.items():
        # one-hot：目标 token 取 1，其他候选取 0。
        target_value = 1.0 if candidate_token == target_token else 0.0
        gradient = probability - target_value
        # 梯度下降：新参数 = 旧参数 - 学习率 * 梯度。
        logits_table[current_token][candidate_token] -= learning_rate * gradient

    return loss


before = probabilities_for(trained_logits, "Agent")
loss_1 = train_step(trained_logits, "Agent", "调用", learning_rate=0.5)
after = probabilities_for(trained_logits, "Agent")

before_summary = {}
after_summary = {}
for token in ["调用", "查询", "维护"]:
    before_summary[token] = round(before[token], 3)
    after_summary[token] = round(after[token], 3)

print("before:", before_summary)
print("loss_1:", round(loss_1, 3))
print("after:", after_summary)

assert after["调用"] > before["调用"]
print("single training step checks passed")


before: {'调用': 0.059, '查询': 0.059, '维护': 0.059}
loss_1: 2.833
after: {'调用': 0.093, '查询': 0.057, '维护': 0.057}
single training step checks passed


### 5.4 多轮训练与 Checkpoint

多轮之间的关系：

```text
epoch 1: 参数 W0 -> 训练全量样本 -> 参数 W1
epoch 2: 参数 W1 -> 训练全量样本 -> 参数 W2
epoch 3: 参数 W2 -> 训练全量样本 -> 参数 W3
```

大模型预训练常用 token 数和 step 数描述进度；epoch 更常见于规模较小的数据集。最大轮数没有固定值。常见停止条件：

1. 达到设定 epoch 数
2. validation loss 不再下降
3. 成本达到预算
4. 评测指标达标
5. 出现过拟合

生产训练通常按固定 step 保存 checkpoint，并用 validation loss 或下游评测选择版本。

In [19]:
import copy  # copy.deepcopy 用于复制完整参数表。


trained_logits = make_empty_logits(vocab)
learning_rate = 0.3
loss_history: list[float] = []
best_checkpoint = None

# 外层循环控制训练轮数。range(100) 会生成 0 到 99。
for epoch in range(1000):
    total_loss = 0.0

    # 内层循环把所有训练对完整使用一遍。
    for current_token, target_token in training_pairs:
        total_loss += train_step(trained_logits, current_token, target_token, learning_rate)

    average_loss = total_loss / len(training_pairs)
    loss_history.append(average_loss)

    # 首轮没有 checkpoint；后续只保存 loss 更低的版本。
    if best_checkpoint is None or average_loss < best_checkpoint["loss"]:
        best_checkpoint = {
            "epoch": epoch + 1,
            "loss": average_loss,
            # deepcopy 创建独立副本，后续训练不会改动已保存的权重。
            "weights": copy.deepcopy(trained_logits),
        }

    # % 是取余运算。每 5 轮打印一次，避免输出过多。
    if (epoch + 1) % 5 == 0:
        print(f"epoch={epoch + 1:02d}, average_loss={average_loss:.3f}")

assert loss_history[-1] < loss_history[0]
assert best_checkpoint["epoch"] == 1000
print("best checkpoint:", {
    "epoch": best_checkpoint["epoch"],
    "loss": round(best_checkpoint["loss"], 3),
})
print("training loop checks passed")


epoch=05, average_loss=1.557
epoch=10, average_loss=0.865
epoch=15, average_loss=0.573
epoch=20, average_loss=0.442
epoch=25, average_loss=0.374
epoch=30, average_loss=0.335
epoch=35, average_loss=0.309
epoch=40, average_loss=0.292
epoch=45, average_loss=0.279
epoch=50, average_loss=0.269
epoch=55, average_loss=0.262
epoch=60, average_loss=0.255
epoch=65, average_loss=0.250
epoch=70, average_loss=0.246
epoch=75, average_loss=0.243
epoch=80, average_loss=0.240
epoch=85, average_loss=0.237
epoch=90, average_loss=0.235
epoch=95, average_loss=0.233
epoch=100, average_loss=0.231
epoch=105, average_loss=0.229
epoch=110, average_loss=0.228
epoch=115, average_loss=0.226
epoch=120, average_loss=0.225
epoch=125, average_loss=0.224
epoch=130, average_loss=0.223
epoch=135, average_loss=0.222
epoch=140, average_loss=0.221
epoch=145, average_loss=0.221
epoch=150, average_loss=0.220
epoch=155, average_loss=0.219
epoch=160, average_loss=0.219
epoch=165, average_loss=0.218
epoch=170, average_loss=0.217

### 5.5 训练产物

一个可加载的模型目录通常包含：

```text
model/
├── config.json                 # 层数、hidden size、词表大小等结构配置
├── tokenizer.json              # 分词规则和词表
├── tokenizer_config.json       # tokenizer 配置
├── model-00001-of-0000N.*      # 模型权重分片
├── model.safetensors.index.json
└── generation_config.json      # 默认生成参数
```

训练 checkpoint 还会保存 optimizer、scheduler、随机数状态和当前 step。部署推理时通常不加载这些训练状态。

### 5.6 用户输入如何变成下一个 token

真实 LLM 推理链路：

流程：

```text
文本 -> tokenizer 编码为 token ID
-> embedding + Transformer
-> 最后位置的 hidden state
-> LM Head 输出词表 logits
-> sampling 得到下一个 token ID
-> tokenizer 解码
```

下面的最长词表匹配只服务于 toy bigram。它将文本转成 toy vocab 中的 token，再读取最后一个 token 对应的参数行。

In [20]:
def tokenize_with_vocab(text: str, vocab: list[str]) -> list[str]:
    # 长 token 排在前面，保证“AI Agent”优先匹配完整词而非短片段。
    sorted_vocab = sorted(vocab, key=len, reverse=True)
    matched_tokens: list[str] = []
    position = 0

    # position 是当前扫描到的字符下标。
    while position < len(text):
        if text[position].isspace():
            position += 1
            continue

        # 从当前位置开始，查找第一个能匹配上的词表 token。
        matched_token = None
        for token in sorted_vocab:
            if text.startswith(token, position): # 从position位置开始跟token匹配
                matched_token = token
                break
        if matched_token is None:
            position += 1
            continue

        matched_tokens.append(matched_token)
        position += len(matched_token)

    return matched_tokens


user_input = "AI No Agent"
matched_tokens = tokenize_with_vocab(user_input, vocab)
# 索引 -1 表示列表最后一个元素。Toy bigram 只使用最后一个 token。
current_token = matched_tokens[-1]
next_probs_after_training = probabilities_for(trained_logits, current_token)
# [:5] 是切片，只保留排序后的前 5 个元素。
top_candidates = sorted(
    next_probs_after_training.items(),
    key=probability_value,
    reverse=True,
)[:5]

print("user_input:", user_input)
print("matched_tokens:", matched_tokens)
print("current_token:", current_token)
print("top next-token candidates:")
for token, probability in top_candidates:
    print(f"  {token}: {probability:.3f}")

assert matched_tokens == ["AI", "Agent"]
best_candidate_token, best_candidate_probability = top_candidates[0]
assert best_candidate_token in {"调用", "查询", "维护"}
assert next_probs_after_training["调用"] > 0.1
assert next_probs_after_training["查询"] > 0.1
assert next_probs_after_training["维护"] > 0.1

print("toy input-to-output checks passed")


user_input: AI No Agent
matched_tokens: ['AI', 'Agent']
current_token: Agent
top next-token candidates:
  维护: 0.369
  查询: 0.330
  调用: 0.299
  AI: 0.000
  Agent: 0.000
toy input-to-output checks passed


### 5.7 使用训练后的 Toy 参数续写

训练完成后，`trained_logits` 保存了当前 token 到候选 token 的分数。推理期间固定参数，循环执行以下步骤：

```text
读取当前 token
-> 从 trained_logits 取得候选分数
-> softmax 转成概率
-> 选择概率最高的 token
-> 追加到上下文
-> 继续预测
```

下面从 `会议` 开始续写 3 个 token。预期结果来自训练语料中的参数关系：`会议 -> 纪要 -> 需要 -> 整理`。

In [21]:
def continue_with_trained_model(
    prompt: str,
    logits_table: dict[str, dict[str, float]],
    vocab: list[str],
    max_new_tokens: int,
    stop_tokens: set[str] | None = None,
) -> tuple[list[str], list[str]]:
    # set[str] | None 表示参数可以是字符串集合，也可以不传。
    # 返回值包含完整上下文和本轮新生成的 token。
    context = tokenize_with_vocab(prompt, vocab)
    if not context:
        raise ValueError("prompt 中没有词表内的 token")

    generated: list[str] = []
    # 没有传 stop_tokens 时，创建空集合。
    stop_tokens = stop_tokens or set()

    for step in range(max_new_tokens):
        # 每轮读取上下文最后一个 token，预测它后面的 token。
        current_token = context[-1]
        probabilities = probabilities_for(logits_table, current_token)
        # 按概率从高到低排序，只展示前三名。
        top_candidates = sorted(
            probabilities.items(),
            key=probability_value,
            reverse=True,
        )[:3]  # 切片保留前三名。
        # top_candidates[0] 是 (token, probability)，这里一次解包成两个变量。
        next_token, next_token_probability = top_candidates[0]

        generated.append(next_token)
        context.append(next_token)

        print(
            f"step {step + 1}: {current_token} -> {next_token}"
            f" (probability={next_token_probability:.3f})"
        )
        top_candidate_summary = {}
        for token, probability in top_candidates:
            top_candidate_summary[token] = round(probability, 3)
        print("  top candidates:", top_candidate_summary)
        print("  context:", context)

        if next_token in stop_tokens:
            break

    return context, generated


continued_tokens, generated_tokens = continue_with_trained_model(
    prompt="会议",
    logits_table=trained_logits,
    vocab=vocab,
    max_new_tokens=3,
)

print("continued text:", " ".join(continued_tokens))

assert generated_tokens == ["纪要", "需要", "整理"]
assert continued_tokens == ["会议", "纪要", "需要", "整理"]
print("trained-model continuation checks passed")


step 1: 会议 -> 纪要 (probability=0.998)
  top candidates: {'纪要': 0.998, 'AI': 0.0, 'Agent': 0.0}
  context: ['会议', '纪要']
step 2: 纪要 -> 需要 (probability=0.997)
  top candidates: {'需要': 0.997, 'AI': 0.0, 'Agent': 0.0}
  context: ['会议', '纪要', '需要']
step 3: 需要 -> 整理 (probability=0.997)
  top candidates: {'整理': 0.997, 'AI': 0.0, 'Agent': 0.0}
  context: ['会议', '纪要', '需要', '整理']
continued text: 会议 纪要 需要 整理
trained-model continuation checks passed


当前边界：

1. Toy bigram 只看当前 token。
2. 训练和推理必须使用同一套 tokenizer 和 vocab。
3. 真实 LLM 通过多层 Transformer 处理完整可见上下文。
4. 真实训练使用反向传播、优化器、GPU、分布式训练。
5. 真实模型参数是矩阵；这里用字典模拟参数表。
6. 推理阶段通常不更新模型权重。RAG、工具结果、对话历史只改变上下文。

## 实验 6：上下文窗口

context window = 输入 token + 输出 token 的总预算。

工程影响：

1. 超长文档需要 chunk。
2. 长对话需要摘要或裁剪。
3. RAG 需要控制召回内容数量。
4. 工具结果需要压缩。
5. prompt、历史消息、检索内容、输出都占 token。

裁剪需要明确优先级。system 和当前问题必须保留；检索证据通常高于较早历史。

In [22]:
def fit_context(parts: list[dict], max_tokens: int) -> list[dict]:
    # selected 保存 (原始位置, 内容块)，便于最后恢复 prompt 顺序。
    selected: list[tuple[int, dict]] = []
    used = 0

    # enumerate(parts) 产生 (原始位置, 内容块)。
    indexed_parts = list(enumerate(parts))

    def priority_of(indexed_part: tuple[int, dict]) -> int:
        return indexed_part[1]["priority"]

    # 优先级高的内容先占用 token 预算。
    ranked_parts = sorted(indexed_parts, key=priority_of, reverse=True)

    for original_order, part in ranked_parts:
        part_size = len(part["tokens"])
        if used + part_size <= max_tokens:
            selected.append((original_order, part))
            used += part_size
        elif part["required"]:
            raise ValueError(f"required context does not fit: {part['name']}")

    def original_order_of(selected_item: tuple[int, dict]) -> int:
        return selected_item[0]

    # 选择完成后，按原始顺序重新组装 prompt。
    selected_in_original_order = sorted(selected, key=original_order_of)
    result = []
    for original_order, part in selected_in_original_order:
        result.append(part)
    return result


context_parts = [
    {
        "name": "system",
        "tokens": simple_tokenize("你是一个严谨的 AI Agent 工程助手。"),
        "priority": 100,
        "required": True,
    },
    {
        "name": "history",
        "tokens": simple_tokenize("用户之前询问过会议纪要和待办提取。"),
        "priority": 40,
        "required": False,
    },
    {
        "name": "retrieval",
        "tokens": simple_tokenize("会议纪要包含议题、决策、负责人和截止时间。"),
        "priority": 80,
        "required": False,
    },
    {
        "name": "question",
        "tokens": simple_tokenize("请提取待办事项。"),
        "priority": 90,
        "required": True,
    },
]

max_context_tokens = 45
selected_parts = fit_context(context_parts, max_tokens=max_context_tokens)
selected_summary = []
total_selected_tokens = 0
for part in selected_parts:
    print("part:", part)
    part_size = len(part["tokens"])
    selected_summary.append((part["name"], part_size))
    total_selected_tokens += part_size

print(selected_summary)
print("total_tokens =", total_selected_tokens)


part: {'name': 'system', 'tokens': ['你', '是', '一', '个', '严', '谨', '的', 'AI', 'Agent', '工', '程', '助', '手', '。'], 'priority': 100, 'required': True}
part: {'name': 'retrieval', 'tokens': ['会', '议', '纪', '要', '包', '含', '议', '题', '、', '决', '策', '、', '负', '责', '人', '和', '截', '止', '时', '间', '。'], 'priority': 80, 'required': False}
part: {'name': 'question', 'tokens': ['请', '提', '取', '待', '办', '事', '项', '。'], 'priority': 90, 'required': True}
[('system', 14), ('retrieval', 21), ('question', 8)]
total_tokens = 43


In [23]:
selected_names = []
for part in selected_parts:
    selected_names.append(part["name"])

# 预算不能超限，必需内容和检索证据必须保留，低优先级历史应被丢弃。
assert total_selected_tokens <= max_context_tokens
assert "system" in selected_names
assert "question" in selected_names
assert "retrieval" in selected_names
assert "history" not in selected_names

print("context window checks passed")


context window checks passed


## 幻觉来源

常见来源：

1. 上下文缺少事实依据。
2. 检索内容错误或过时。
3. prompt 要求模型强行回答。
4. 采样随机性较高。
5. 任务需要工具查询，模型直接编造。
6. 长上下文里关键证据被截断。
7. 输出格式约束掩盖了事实错误。

工程处理：

1. 需要事实的问题接 RAG 或工具。
2. 输出引用来源。
3. 高风险结论加人审。
4. 对关键字段做校验。
5. 用 eval dataset 记录失败样例。

## 本章要点

1. tokenizer 把文本编码为 token ID，embedding 把 token ID 转成向量。
2. 位置信息让模型区分相同 token 在不同位置上的含义。
3. Transformer 通过 Attention、FFN、残差和归一化计算 hidden state。
4. 最后一个已有位置的 hidden state 经过 LM Head 后产生下一个 token 的 logits。
5. Attention 按位置计算权重，并用 causal mask 屏蔽未来位置。
6. temperature 调整概率分布，sampling 选出下一个 token。
7. prefill 处理 prompt，decode 逐 token 生成，KV Cache 缓存历史 Key 和 Value。
8. 训练通过 loss、梯度和 optimizer 更新参数，并用 validation 与 checkpoint 管理过程。
9. 推理加载固定参数，将每个新 token 追加到上下文。
10. 上下文裁剪必须保留 system 和当前问题，并按任务价值选择证据与历史。
11. 幻觉需要通过上下文、工具、引用、校验、评测和人审处理。

## 下一步

第 3 章进入 Prompt 与结构化输出。

重点：

1. system prompt
2. 输入协议
3. few-shot
4. JSON Schema
5. 输出校验
6. prompt regression